<a href="https://colab.research.google.com/github/Siddesh-2004/FindYourPhone/blob/main/featureBasedRecommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Imports**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Load & Clean
# ═══════════════════════════════════════════════════════════════
df = pd.read_csv("/content/phones.csv")

# Drop duplicates
df.drop_duplicates(inplace=True)

# Drop dead columns
dead_cols = ["img", "display pixels", "display frequency (in Hz)",
             "Punch Hole", "extended_upto", "core", "os_version"]
df.drop(columns=dead_cols, inplace=True)

# Extract numeric fields
df["ram"]            = df["ram"].str.extract(r"(\d+)").astype(float).astype("Int64")
df["storage"]        = df["ram (inbuilt)"].str.extract(r"(\d+)").astype(float).astype("Int64")
df["battery"]        = df["battery (in mAh)"].str.extract(r"(\d+)").astype(float).astype("Int64")
df["display_size"]   = df["display size"].str.extract(r"([\d.]+)").astype(float)
df["fast_charging"]  = df["fast charging"].str.extract(r"(\d+)").astype(float).astype("Int64")
df["frequency"]      = df["frequency"].str.extract(r"([\d.]+)").astype(float)
df["front_camera_mp"]= df["front_camera"].str.extract(r"(\d+)").astype(float).astype("Int64")
df["rear_camera_mp"] = df["rear_camera"].str.extract(r"(\d+)").astype(float).astype("Int64")

df.drop(columns=["ram (inbuilt)", "battery (in mAh)", "display size",
                  "fast charging", "front_camera", "rear_camera"], inplace=True)

# Fix booleans
for col in ["4G", "5G", "NFC", "VoLTE"]:
    df[col] = df[col].map(lambda x: 1 if str(x).strip().lower() == "true" else 0).astype("Int64")

# Clean os_brand
valid_os = {"Android", "iOS", "HarmonyOS"}
df["os_brand"] = df["os_brand"].apply(lambda x: x if x in valid_os else None)

print("After cleaning:", df.shape)



After cleaning: (1018, 21)


In [ ]:
features = ['ram', 'storage', 'battery', 'fast_charging',
            'front_camera_mp', 'rear_camera_mp', 'specs_score']

# Cap outliers via IQR
for col in features:
    df[col] = df[col].astype(float)  # Int64 doesn't support float clip bounds
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    df[col] = df[col].clip(lower=q1 - 1.5 * iqr, upper=q3 + 1.5 * iqr)

# Drop low-variance columns confirmed from variance check
df.drop(columns=["display_size", "rating"], inplace=True, errors="ignore")

print("After variance step:", df.shape)


After variance step: (1018, 19)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Normalize Features
# ═══════════════════════════════════════════════════════════════
df_model = df.dropna(subset=features).reset_index(drop=True)

scaler = MinMaxScaler()
df_model[features] = scaler.fit_transform(df_model[features])

print("After normalization:", df_model.shape)



After normalization: (791, 19)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Rule-Based Filter
# ═══════════════════════════════════════════════════════════════
def rule_based_filter(df, budget=None, os_brand=None, need_5g=False, need_nfc=False):
    filtered = df.copy()
    if budget is not None:
        filtered = filtered[filtered["price"] <= budget]
    if os_brand is not None:
        filtered = filtered[filtered["os_brand"] == os_brand]
    if need_5g:
        filtered = filtered[filtered["5G"] == 1]
    if need_nfc:
        filtered = filtered[filtered["NFC"] == 1]
    return filtered.reset_index(drop=True)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Cosine Similarity Scorer
# ═══════════════════════════════════════════════════════════════
def build_user_vector(ram=8, storage=128, battery=4500, fast_charging=33,
                      front_camera_mp=16, rear_camera_mp=50, specs_score=60):
    raw = pd.DataFrame([[ram, storage, battery, fast_charging,
                         front_camera_mp, rear_camera_mp, specs_score]],
                       columns=features)
    return scaler.transform(raw)


def score_phones(candidate_df, user_vector):
    phone_matrix = candidate_df[features].values
    scores = cosine_similarity(user_vector, phone_matrix)[0]
    result = candidate_df.copy()
    result["match_score"] = (scores * 100).round(2)
    return result.sort_values("match_score", ascending=False).reset_index(drop=True)



In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — recommend() function
# ═══════════════════════════════════════════════════════════════
def recommend(budget=None, os_brand=None, need_5g=False, need_nfc=False,
              ram=8, storage=128, battery=4500, fast_charging=33,
              front_camera_mp=16, rear_camera_mp=50, specs_score=60, top_n=10):

    candidates = rule_based_filter(df_model, budget=budget, os_brand=os_brand,
                                   need_5g=need_5g, need_nfc=need_nfc)
    if candidates.empty:
        print("No phones match your filters.")
        return pd.DataFrame()

    user_vector = build_user_vector(ram=ram, storage=storage, battery=battery,
                                    fast_charging=fast_charging,
                                    front_camera_mp=front_camera_mp,
                                    rear_camera_mp=rear_camera_mp,
                                    specs_score=specs_score)

    ranked = score_phones(candidates, user_vector)
    return ranked[["name", "price", "match_score"]].head(top_n)



In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — 3 Persona Demos
# ═══════════════════════════════════════════════════════════════
personas = [
    {
        "name": "Student",
        "params": dict(budget=12000, os_brand="Android", need_5g=False,
                       ram=4, storage=64, battery=5000, fast_charging=18,
                       front_camera_mp=8, rear_camera_mp=13, specs_score=40)
    },
    {
        "name": "Photographer",
        "params": dict(budget=50000, os_brand="Android", need_5g=False,
                       ram=8, storage=256, battery=4500, fast_charging=33,
                       front_camera_mp=32, rear_camera_mp=108, specs_score=70)
    },
    {
        "name": "Gamer",
        "params": dict(budget=40000, os_brand="Android", need_5g=True,
                       ram=12, storage=256, battery=5000, fast_charging=65,
                       front_camera_mp=16, rear_camera_mp=50, specs_score=80)
    },
]

for p in personas:
    print(f"\n=== {p['name']} ===")
    print(recommend(**p["params"]).to_string(index=False))



=== Student ===
                          name  price  match_score
                      Vivo Y02   8999        83.19
                 Lava Yuva Pro   7799        82.98
     Vivo Y02 (2GB RAM + 32GB)   7999        81.27
                   Realme C30s   6499        75.72
                 itel Vision 3   7199        74.21
           itel Vision 3 Turbo   7100        74.21
itel Vision 3 (2GB RAM + 32GB)   5849        74.06
               Xiaomi Redmi A1   5899        73.67
                      Poco C50   5749        73.67
  Realme C30s (4GB RAM + 64GB)   7999        72.63

=== Photographer ===
                                   name  price  match_score
                        Oppo Reno 8T 4G  19999        96.35
     Vivo V23 Pro 5G (12GB RAM + 256GB)  35994        95.37
Samsung Galaxy A73 5G (8GB RAM + 256GB)  44999        95.34
                               Vivo V28  28990        95.22
                               Honor X8  16999        94.96
                        Vivo V23 Pro 5G 